<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;"><img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;"></div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">MSc. CORO DASSIP</div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Camera Calibration — Algorithm</b></h1>
</div>


## Purpose

This notebook is the **step-by-step coding algorithm** for `main.ipynb`. It tells the reader exactly **what to code, in what order, what each step must produce, and what must be checked before continuing**.

## Algorithm Rules

- Follow the numbered steps from **0. Start** to **End** without skipping dependencies.
- Create variables, helper functions, loops, conditions, metrics, figures, and validation checks in the order specified.
- Reuse validated intermediate results; do not silently recreate different versions later.
- Stop on missing data, invalid shapes, non-finite numerical results, or missing required artifacts.
- `main.ipynb` is the executable implementation of this algorithm.

## End-to-End Flow

```text
Start → setup → image discovery → corner detection/refinement → planar points → normalization → DLT → homographies → Zhang constraints → intrinsics → poses → reprojection → errors → figures → validation → End
```

## Inputs and Final Outputs

| Item | Description |
| --- | --- |
| **Inputs** | Calibration images, 8×6 internal-corner geometry, 0.03 m square size |
| **Final outputs** | Intrinsic matrix K, one (R,t) pose per retained view, reprojection metrics, six diagnostic figures |
| **Implementation target** | `main.ipynb` |


## 0. Start

**Coding instructions**

1. Open the lab with the project environment selected in VS Code.
2. Treat this notebook as the coding plan for `main.ipynb`; do not compute results here.
3. Proceed only in the order below because later calibration quantities depend on earlier validated geometry.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 1. Create the Setup

**Coding instructions**

1. Import `Path`, OpenCV, NumPy, and Matplotlib.
2. Set NumPy display precision so matrices and residuals are readable.
3. Define square size, 8×6 internal corners, minimum valid views, and corner-refinement criteria.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 2. Create Repository-Relative Paths

**Coding instructions**

1. Create the calibration-image path and `outputs/figures/` path.
2. Create the output directory if needed.
3. Define supported image extensions; use no machine-specific absolute path.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 3. Discover and Validate Calibration Images

**Coding instructions**

1. Enumerate supported image files and sort them deterministically.
2. If no image is found, stop with `FileNotFoundError`.
3. Create containers for readable images and retained calibration views.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 4. Detect and Refine Chessboard Corners

**Coding instructions**

1. For each image: read it and convert it to grayscale.
2. Detect the complete 8×6 chessboard pattern.
3. If found, refine corners with `cornerSubPix`; otherwise reject that view.
4. Store image name, image array, and refined 2-D points.
5. Stop if fewer than the required number of valid views remain.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 5. Build Metric Planar Coordinates

**Coding instructions**

1. Create 48 planar points in the same ordering as detected corners.
2. Set each point to `(x·square_size, y·square_size)` in metres.
3. Keep the board on `Z=0`; add the zero z-coordinate only when 3-D points are needed.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 6. Define Homogeneous-Coordinate Conversion

**Coding instructions**

1. Create a helper that appends a homogeneous coordinate of 1 to every 2-D point.
2. Use it for both plane and image correspondences.
3. Validate the returned shape before matrix computations.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 7. Normalize Plane and Image Points

**Coding instructions**

1. Compute each point-set centroid and shift to zero mean.
2. Compute mean Euclidean distance from the centroid.
3. Set scale to `sqrt(2)/mean_distance`; reject coincident-point configurations.
4. Build each 3×3 normalization transform and apply it.
5. Store `T_plane`, `T_image`, and normalized point sets per view.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 8. Build the Normalized DLT System

**Coding instructions**

1. For every correspondence add the two standard DLT rows to matrix `Q`.
2. Solve `Qh=0` by SVD.
3. Select the right-singular vector for the smallest singular value.
4. Reshape it into a normalized 3×3 homography.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 9. Denormalize Every Homography

**Coding instructions**

1. Compute `H = inv(T_image) @ H_normalized @ T_plane`.
2. Normalize by `H[2,2]`; reject numerically degenerate scale.
3. Store one finite 3×3 homography per retained view.
4. Recheck that each view still contains 48 plane/image correspondences.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 10. Build Zhang Multi-View Constraints

**Coding instructions**

1. Extract `h1` and `h2` from every homography.
2. Implement the Zhang `v_ij` vector.
3. Stack `v12` and `v11-v22` for all views into matrix `V`.
4. Solve `Vb=0` by SVD and retain the last right-singular vector `b`.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 11. Recover the Intrinsic Matrix

**Coding instructions**

1. Unpack `b` into the six symmetric-conic coefficients.
2. Compute the denominator and reject a degenerate intrinsic system.
3. Recover `v0`, `lambda`, `alpha`, `beta`, `gamma`, and `u0`.
4. If focal-length terms are invalid, flip the scale sign of `b` and recompute.
5. Assemble `K` and verify finite positive focal lengths.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 12. Recover One Camera Pose per View

**Coding instructions**

1. Compute `K^-1 h1`, `K^-1 h2`, and `K^-1 h3`.
2. Use the norm of `K^-1 h1` for the common pose scale.
3. Form `r1`, `r2`, `r3=r1×r2`, and `t`.
4. Project the approximate rotation to the nearest orthonormal matrix with SVD.
5. If `det(R)<0`, correct the sign so `R` is a proper rotation.
6. Store finite `R` and `t`.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 13. Reproject Calibration Points

**Coding instructions**

1. Convert planar points to `(X,Y,0)`.
2. Transform them into the camera frame with `R` and `t`.
3. Project with `K` and divide by homogeneous depth.
4. Reject near-zero depth and store the projected 2-D points.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 14. Compute Calibration Errors

**Coding instructions**

1. Compute residual vectors `projected-observed`.
2. Convert residuals to point-wise Euclidean errors.
3. Compute per-view mean error and RMSE.
4. Concatenate all errors and compute overall mean error and RMSE.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 15. Generate the Required Diagnostics

**Coding instructions**

1. Save the homography-estimation pipeline figure.
2. Save detected corners, camera poses, and reprojection overlays.
3. Save mean reprojection error by view and the global error distribution.
4. Build every plot from the same validated arrays used for numerical metrics.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## 16. Run Final Validation

**Coding instructions**

1. Verify `K` is finite and 3×3.
2. For every view verify finite `H`, orthonormal proper `R`, finite `t`, and finite residuals.
3. Verify all six declared output files exist.
4. Print the final validation-pass message only after all checks succeed.

**Checkpoint:** Do not continue until the outputs created in this step are valid and available to the next step.


## Complete End-to-End Pseudocode

```text
0. Start
1. Create the Setup
2. Create Repository-Relative Paths
3. Discover and Validate Calibration Images
4. Detect and Refine Chessboard Corners
5. Build Metric Planar Coordinates
6. Define Homogeneous-Coordinate Conversion
7. Normalize Plane and Image Points
8. Build the Normalized DLT System
9. Denormalize Every Homography
10. Build Zhang Multi-View Constraints
11. Recover the Intrinsic Matrix
12. Recover One Camera Pose per View
13. Reproject Calibration Points
14. Compute Calibration Errors
15. Generate the Required Diagnostics
16. Run Final Validation
17. End
```


## Task Coverage

Every task defined in the Problem Statement is covered by the coding sequence above:

- **Task 1:** Load the Sorted JPEG Calibration Images
- **Task 2:** Detect and Refine Chessboard Corners
- **Task 3:** Build the Planar World Coordinates
- **Task 4:** Compute $T_{\mathrm{image}}$ and $T_{\mathrm{plane}}$
- **Task 5:** Build the DLT Matrix $Q$ and Solve $Q\mathbf{h}=0$ by SVD
- **Task 6:** Denormalize Each Homography
- **Task 7:** Build the Zhang Matrix $V$ and Solve $Vb=0$ by SVD
- **Task 8:** Recover $\alpha,\beta,\gamma,u_0,v_0$ and Construct $K$
- **Task 9:** Recover $R$ and $t$ for Every Retained View
- **Task 10:** Reproject the $Z=0$ Calibration Points
- **Task 11:** Compute Point-wise Errors, Mean Error and RMSE
- **Task 12:** Produce and Save the Six Required Diagnostic Figures
- **Task 13:** Run the Numerical and Output-file Validation Checks


## Notebook Relationship

- `problem_statement.ipynb` defines **what must be solved**.
- `theory.ipynb` explains **why the methods work**.
- `requirements.ipynb` defines **what is needed to run the lab**.
- `algorithm.ipynb` defines **how to build the solution step by step**.
- `main.ipynb` is the **executable implementation** of this algorithm.


## 17. End

The coding plan is complete. Open `main.ipynb` and implement or verify the workflow from top to bottom in the same dependency order. The lab is complete only when the final validation stage passes and every required output exists.
